# 09 · Numba CUDA ②: 히스토그램 (atomic · 공유메모리)

> **CuPy 2일 집중 코스 — Day 2 / 단원 6 (Numba CUDA 커널 작성, GTC 07 기반)**

히스토그램으로 **데이터 레이스 → atomic → 공유메모리**의 최적화 단계를 직접 구현합니다.
07에서 배운 **동기화/atomic(6절)·공유메모리(4절)** 개념이 실제로 어떻게 쓰이는지 봅니다.

## 이 노트북에서 구현하는 개념 (07 참조)
- **6. 동기화/atomic**: 데이터 레이스 → `cuda.atomic.add` → `cuda.syncthreads`
- **4. 메모리 계층**: 블록별 **공유 메모리(`cuda.shared.array`)** 로 전역 경합 감소

## 학습 목표
- 병렬 누적의 **데이터 레이스**를 이해하고 `atomic`으로 고친다.
- **공유메모리 privatization**으로 전역 atomic 경합을 줄여 가속한다.

## 목차
1. [히스토그램 & 전역 naive(레이스)](#1)
2. [데이터 레이스 진단](#2)
3. [atomic으로 수정](#3)
4. [공유메모리 최적화](#4)
5. [성능 비교](#5)
6. [(선택) cooperative & ncu](#6)
7. [체크포인트](#7)

> 필요: `numba`. 데이터는 자체 생성(난수 바이트). GTC 원본은 책 텍스트(문자 빈도)를 사용합니다.

In [ ]:
import numpy as np, cupy as cp, numba
from numba import cuda
from course_utils import print_env, bench, gpu_ms, print_bench
print_env()

BINS = 256
N = 1 << 24
values = cp.random.randint(0, BINS, size=N, dtype=cp.int32)  # 0..255 값
ref = np.bincount(cp.asnumpy(values), minlength=BINS)        # 정답(CPU)

<a id="1"></a>
## 1. 히스토그램 & 전역 naive(레이스)

히스토그램 = 각 값의 등장 횟수 세기. 가장 단순한 구현: 각 스레드가 자기 값의 칸을 +1.
하지만 `hist[v] = hist[v] + 1` 은 **읽기-수정-쓰기**라 여러 스레드가 같은 칸을 동시에 건드리면 깨집니다.

In [ ]:
@cuda.jit
def hist_naive(values, hist):
    i = cuda.grid(1)
    if i < values.size:
        v = values[i]
        hist[v] = hist[v] + 1      # ⚠️ 데이터 레이스

threads = 256; blocks = (N + threads - 1)//threads
hist = cp.zeros(BINS, dtype=cp.int32)
hist_naive[blocks, threads](values, hist)
cp.cuda.Device().synchronize()
print('naive 합계:', int(hist.sum()), '/ 기대:', N)   # 합계가 모자람!

<a id="2"></a>
## 2. 데이터 레이스 진단

두 스레드가 같은 칸을 동시에 갱신하면:
1) T0이 count(0) 읽음 → 2) T1도 count(0) 읽음 → 3) 둘 다 1로 써서 **한 증가가 사라짐**.
그래서 위 합계가 N보다 작습니다. 해결: **나눌 수 없는 atomic 연산**.

<a id="3"></a>
## 3. atomic으로 수정 — 연습

`cuda.atomic.add(array, index, value)` 는 `array[index] += value` 를 **원자적**으로 수행합니다(6절).

In [ ]:
@cuda.jit
def hist_atomic(values, hist):
    i = cuda.grid(1)
    if i < values.size:
        # TODO: cuda.atomic.add(hist, values[i], 1) 로 안전하게 증가
        pass

hist = cp.zeros(BINS, dtype=cp.int32)
# hist_atomic[blocks, threads](values, hist); cp.cuda.Device().synchronize()
# assert int(hist.sum())==N; np.testing.assert_array_equal(cp.asnumpy(hist), ref); print('atomic OK')

<details><summary>💡 해답 보기</summary>

```python
@cuda.jit
def hist_atomic(values, hist):
    i = cuda.grid(1)
    if i < values.size:
        cuda.atomic.add(hist, values[i], 1)

hist = cp.zeros(BINS, dtype=cp.int32)
hist_atomic[blocks, threads](values, hist); cp.cuda.Device().synchronize()
assert int(hist.sum()) == N
np.testing.assert_array_equal(cp.asnumpy(hist), ref); print('atomic OK')
```
</details>

<a id="4"></a>
## 4. 공유메모리 최적화

전역 메모리 atomic은 **모든 블록이 같은 256칸을 두고 경합**해 느립니다.
각 블록이 **공유메모리에 자기만의 히스토그램**(privatization)을 만들어 거기서 atomic을 한 뒤,
마지막에 블록 결과만 전역에 합칩니다. 공유메모리는 온칩이라 빠르고 경합 범위가 블록 내로 줄어듭니다(4절).

In [ ]:
@cuda.jit
def hist_shared(values, hist):
    smem = cuda.shared.array(256, numba.int32)   # 블록별 히스토그램
    t = cuda.threadIdx.x; nt = cuda.blockDim.x
    j = t
    while j < 256:                # 공유메모리 0으로 초기화
        smem[j] = 0; j += nt
    cuda.syncthreads()            # 초기화 완료까지 대기
    i = cuda.grid(1); stride = cuda.gridsize(1)
    while i < values.size:        # 블록 내 atomic (경합 ↓)
        cuda.atomic.add(smem, values[i], 1); i += stride
    cuda.syncthreads()            # 집계 완료까지 대기
    j = t
    while j < 256:                # 블록 결과를 전역에 합치기
        cuda.atomic.add(hist, j, smem[j]); j += nt

hist = cp.zeros(BINS, dtype=cp.int32)
hist_shared[1024, threads](values, hist); cp.cuda.Device().synchronize()
np.testing.assert_array_equal(cp.asnumpy(hist), ref); print('shared OK')

<a id="5"></a>
## 5. 성능 비교

전역 atomic vs 공유메모리 atomic 을 비교합니다. 값 분포가 좁을수록(경합 심할수록) 공유메모리 이득이 큽니다.

In [ ]:
def run_atomic():
    hist[:] = 0; hist_atomic[blocks, threads](values, hist)
def run_shared():
    hist[:] = 0; hist_shared[1024, threads](values, hist)
print_bench(bench(run_atomic, n_repeat=20, n_warmup=5, name='global atomic'))
print_bench(bench(run_shared, n_repeat=20, n_warmup=5, name='shared atomic'))
print('speedup:', round(gpu_ms(bench(run_atomic))/gpu_ms(bench(run_shared)), 2))

<a id="6"></a>
## 6. (선택) cooperative load & Nsight Compute

<details><summary>펼쳐 보기 — 더 빠른 로드와 프로파일 </summary>

**cooperative groups**(`cuda.cooperative`)의 `coop.block.load`로 값을 **striped(coalesced)** 로 한 번에 읽어
로드와 갱신을 분리하면 더 빨라집니다. (실험적 API — 버전에 따라 인터페이스가 다를 수 있음)

```python
import cuda.cooperative.experimental as coop
block_load = coop.block.load(numba.int32, threads_per_block, items_per_thread, 'striped')
@cuda.jit(link=block_load.files)
def hist_coop(values, hist):
    items = cuda.local.array(items_per_thread, numba.int32)
    block_load(values, items)        # 연속 로드
    ...  # 공유메모리 히스토그램 갱신
```

**Nsight Compute**: 단계별로 `ncu`를 돌려 **Memory Workload** 처리량을 비교하면, 공유메모리 버전의 전역 트랜잭션이 줄어든 것을 확인할 수 있습니다(08의 ncu 워크플로와 동일).
</details>

## 🧪 추가 연습 & 비교

**비교 — `cp.bincount` 대비**: CuPy 내장 히스토그램과 정확성·속도를 비교하세요.

In [ ]:
def run_shared():
    h = cp.zeros(BINS, dtype=cp.int32); hist_shared[1024, threads](values, h); return h
h_cp = cp.bincount(values, minlength=BINS)
np.testing.assert_array_equal(cp.asnumpy(run_shared()), cp.asnumpy(h_cp)); print('shared == bincount')
print_bench(bench(lambda: cp.bincount(values, minlength=BINS), n_repeat=20, name='cp.bincount'))

**연습 — grid-stride + items_per_thread**: 각 스레드가 여러 값을 처리하도록 공유메모리 히스토그램을 확장하세요(블록 수↓, 처리량↑).
힌트: grid-stride 루프(`while i < n: ...; i += stride`)는 이미 적용돼 있습니다. 블록 수를 줄여(예: 256) 스레드당 처리량을 늘려 측정해 보세요.

In [ ]:
for nblocks in [256, 1024, 4096]:
    def run(b=nblocks):
        h = cp.zeros(BINS, dtype=cp.int32); hist_shared[b, threads](values, h)
    print('blocks', nblocks, '->', round(gpu_ms(bench(run, n_repeat=20, n_warmup=5)), 4), 'ms')

<a id="7"></a>
## 7. 체크포인트

- [ ] 병렬 누적의 데이터 레이스를 설명할 수 있다
- [ ] `cuda.atomic.add`로 안전하게 히스토그램을 만들었다
- [ ] 공유메모리 privatization + `syncthreads`로 가속했다
- [ ] 전역 vs 공유 atomic 성능을 비교했다

다음: **`10_cccl`** — 직접 커널을 짜는 대신 **검증된 병렬 알고리즘**(reduce/scan/transform)으로 같은 일을 합니다.